# hexmb quick start: a beating left ventricle in five minutes

`hexmb` builds all-hexahedral O-grid meshes and writes them as OpenFOAM cases. This notebook
installs it, meshes a public-domain left ventricle through its 20 cardiac phases, checks every
phase, and shows how to feed in your own surface.

Runs on Google Colab or any Jupyter with Python 3.10+. No OpenFOAM is needed for anything here.

In [ ]:
%pip install -q "hexmb[all] @ git+https://github.com/JieWangnk/hexmb"
import hexmb; print("hexmb", hexmb.__version__)

## 1. The bundled heart, all 20 phases

The package ships the Cardiac Atlas Project model of one Sunnybrook subject (CC0). The demo reads
the endocardial surface at every phase, builds the same conformal split-hex mesh on each, gates it
(every cell volume positive, no degenerate faces) and writes a moving-mesh OpenFOAM case.

In [ ]:
from hexmb.cli import main
main(["demo", "-o", "out_demo", "--render"])

In [ ]:
from IPython.display import Image, display
display(Image("out_demo/demo.png"))

What was written:

- `out_demo/lv_moving/constant/polyMesh`: the mesh at the reference phase (smallest cavity),
  patches `wall`, `inlet` (mitral), `outlet1` (aortic);
- `out_demo/lv_moving/<time>/wantedDisplacement`: one point-displacement field per phase, zero at
  the reference, which OpenFOAM 12's `interpolator` mover reads (`foamRun -solver movingMesh`);
- `out_demo/lv_moving/motion_quality.csv`: the per-phase gate;
- `out_demo/lv_endo_phase1.stl`: the end-diastolic endocardial surface as a CAD file.

In [ ]:
import csv
with open("out_demo/lv_moving/motion_quality.csv") as f:
    rows = list(csv.DictReader(f))
print(f"{'phase':>5} {'t (s)':>6} {'volume mL':>10} {'min cell vol mm3':>17} {'non-positive':>13}")
for r in rows:
    print(f"{int(r['phase']):5d} {float(r['time_s']):6.3f} {float(r['volume_mL']):10.1f} "
          f"{float(r['min_cell_volume_mm3']):17.3e} {int(r['n_nonpositive']):13d}")

## 2. The same from Python, one phase

The templates take an ordered `(NL, NT, 3)` endocardial grid (mm, apex first) and a frame.

In [ ]:
from importlib import resources
from hexmb.io.cap import cap_endo_surface, surface_frame
from hexmb.templates.conformal_lv import ConformalLVConfig, build_conformal_lv, polymesh_quality

ex = resources.files("hexmb") / "data/cap/SCD0000101/SCD0000101_1.model.exnode"
G = cap_endo_surface(str(ex), NL=40, NT=56)          # (40, 56, 3) grid in mm
axis, e1, e2, mu = surface_frame(G)
cfg = ConformalLVConfig(body_levels=60)              # refine the body axially: skew-safe on a moving LV
mesh = build_conformal_lv(G, axis, e1, e2, cfg, mu=mu)
print(polymesh_quality(mesh))
mesh.write("out_one_phase")                           # constant/polyMesh

## 3. Bring your own ventricle

Any closed or open endocardial surface works: STL, OBJ, PLY, OFF, VTK/VTP, a NIfTI segmentation
mask, or a saved grid. The reader ray-casts the ring grid along the apex-base axis, detecting the
landmarks if you do not give them. Upload a file to the notebook and point at it:

In [ ]:
# the demo's own STL stands in for your file here
from hexmb.io.surface import grid_from_file
G2, axis2, e12, e22, mu2, info = grid_from_file("out_demo/lv_endo_phase1.stl", NL=40, NT=56)
print(info["landmarks"], "apex", info["apex"].round(1), "base", info["base"].round(1))
mesh2 = build_conformal_lv(G2, axis2, e12, e22, mu=mu2)
print(polymesh_quality(mesh2))

In [ ]:
# or from the command line, including per-phase surfaces for a moving mesh:
# !hexmb lv --surface my_lv.stl -o case_lv
# !hexmb lv --surface my_mask.nii.gz --label 1 -o case_lv
# !hexmb motion --surfaces phase_*.vtk --period 0.9 -o case_moving

## 4. Check with OpenFOAM (optional)

With OpenFOAM 12 on the PATH, `hexmb verify -o out_one_phase` runs `checkMesh`, and
`cd out_demo/lv_moving && foamRun -solver movingMesh` moves the mesh through the cycle.